# URM Sudoku Internal Cycle Visualization

This notebook visualizes how the checkpoint evolves a Sudoku prediction across the model's outer loops and internal refinement cycles.

Target command settings:

```bash
python evaluate_trained_model.py \
  --checkpoint checkpoints/URM-sudoku-base \
  --max_problems 4096 \
  --loops 4 \
  --batch_size 4096
```

For this checkpoint, each outer loop runs the internal `L_cycles` refinement steps once, so we plot the Sudoku map after every internal step for every outer loop.

In [ ]:
import numpy as np
import torch
from IPython.display import HTML, display

from sudoku_trace_utils import (
    build_trace_summary,
    display_trace_summary,
    get_batch,
    load_visualization_setup,
    trace_outer_loops,
)

np.set_printoptions(linewidth=140)

In [ ]:
CHECKPOINT = 'checkpoints/URM-sudoku-base'
SPLIT = 'test'
BATCH_SIZE = 4096
BATCH_INDEX = 70
LOOPS = 4
H_CYCLES = None
L_CYCLES = None

setup = load_visualization_setup(
    CHECKPOINT,
    split=SPLIT,
    batch_size=BATCH_SIZE,
    loops=LOOPS,
    h_cycles=H_CYCLES,
    l_cycles=L_CYCLES,
)

batch_info = get_batch(setup['dataloader'], setup['device'], batch_index=BATCH_INDEX)
records = trace_outer_loops(setup['model'], batch_info['batch_gpu'])

resolved_h_cycles = setup['model'].model.inner.config.H_cycles
resolved_l_cycles = setup['model'].model.inner.config.L_cycles

print(f"checkpoint step: {setup['step']}")
print(f"set name: {batch_info['set_name']}")
print(f"global batch size: {batch_info['global_batch_size']}")
print(f"outer loops: {setup['model'].model.config.loops}")
print(f"H_cycles: {resolved_h_cycles}")
print(f"L_cycles: {resolved_l_cycles}")
print(f"trace panels: {len(records)}")

In [ ]:
labels = batch_info['batch_cpu']['labels']
valid_mask = (labels != -100).any(dim=1).cpu().numpy()
valid_indices = np.flatnonzero(valid_mask)

final_preds = records[-1].preds
final_exact = ((final_preds == labels) | (labels == -100)).all(dim=1).cpu().numpy() & valid_mask

print(f'valid problems in this batch: {valid_mask.sum()}')
print(f'final exact solved problems: {final_exact.sum()} / {valid_mask.sum()}')
print('first 10 solved sample indices:', valid_indices[final_exact[valid_indices]][:10])
print('first 10 unsolved sample indices:', valid_indices[~final_exact[valid_indices]][:10])

In [ ]:
SAMPLE_INDEX = 2

summary = build_trace_summary(batch_info['batch_cpu'], records, sample_index=SAMPLE_INDEX)
display_trace_summary(summary, max_cols=4)

## Notes

- `Original Input` shows the real Sudoku puzzle from the dataset. Blue cells are given clues.
- `Model Input` shows what the evaluation code actually fed into the model. For this checkpoint, `masked_input.enabled=true`, so this is a partially masked version of the solution and usually has many more visible digits.
- `loop=1 / H1/L1` ... `loop=4 / H1/L24` show the predicted Sudoku after each internal refinement step across outer loops.
- Green cells are correct predictions for blank cells.
- Red cells are incorrect predictions for blank cells.
- Yellow cells are still unresolved or predicted as non-digit tokens.
- `Target` is the ground-truth solved Sudoku.

To inspect a different puzzle, change `SAMPLE_INDEX` and rerun the last cell.
